In [ ]:

# This file is part of the article:
# "Leveraging Remote Traffic Data for Local Air Pollutant Estimation:
# A Scenario-Based Machine Learning Study Across London Monitoring Sites"
#
# Copyright (C) 2026 The authors
##
# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# This program is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
# GNU General Public License for more details.
#
# You should have received a copy of the GNU General Public License
# along with this program. If not, see <https://www.gnu.org/licenses/>.

In [1]:
import pandas as pd
import os
import numpy as np
import sqlite3
import geopandas as gpd
from shapely.wkt import loads
from pathlib import Path

summary_file = Path("preprocessing_data_counts.csv")

def parse_defra_datetime(df, date_col="Date", time_col="Time"):
    """
    Parse DEFRA date and time columns into a single datetime column.

    Supported date formats:
        - mm-dd-yy
        - mm/dd/yy
        
    The value 24:00:00 is converted to 00:00:00 of the following day.

    Returns
    -------
    pandas.Series
        Datetime values (timezone-naive).
    """

    def parse_date(date_str):
        if pd.isna(date_str):
            return pd.NaT

        date_str = str(date_str).strip()

        if "-" in date_str:
            return pd.to_datetime(
                date_str,
                format="%m-%d-%y",
                errors="coerce"
            )

        # mm/dd/yy or mm/dd/yyyy
        if "/" in date_str:
            try:
                return pd.to_datetime(
                    date_str,
                    format="%m/%d/%y",
                    errors="raise"
                )
            except ValueError:
                return pd.to_datetime(
                    date_str,
                    format="%m/%d/%Y",
                    errors="coerce"
                )

        return pd.NaT

    dates = df[date_col].apply(parse_date)

    if dates.isna().any():
        raise ValueError(
            f"Invalid dates found: "
            f"{df.loc[dates.isna(), date_col].unique().tolist()}"
        )

    time_parts = (
        df[time_col]
        .astype(str)
        .str.strip()
        .str.split(":", expand=True)
    )

    if time_parts.shape[1] != 3:
        raise ValueError(
            "Time must have HH:MM:SS format."
        )

    hours = pd.to_numeric(time_parts[0], errors="coerce")
    minutes = pd.to_numeric(time_parts[1], errors="coerce")
    seconds = pd.to_numeric(time_parts[2], errors="coerce")


    datetime = (
        dates
        + pd.to_timedelta(hours, unit="h")
        + pd.to_timedelta(minutes, unit="m")
        + pd.to_timedelta(seconds, unit="s")
    )

    return datetime

def load_and_convert_traffic_to_utc(df):
    df = df.copy()

    df["datetime_local_naive"] = pd.to_datetime(
        df["date"].astype(str).str.strip()
        + " "
        + df["time"].astype(str).str.strip(),
        errors="coerce"
    )

    try:
        df["datetime_london"] = (
            df["datetime_local_naive"]
            .dt.tz_localize(
                "Europe/London",
                ambiguous="infer",
                nonexistent="NaT"
            )
        )

    except Exception as exc:
        raise ValueError(
            "Unable to automatically infer ambiguous timestamps during "
            "the BST-to-GMT transition. Verify that the records remain "
            "in their original chronological order."
        ) from exc

    df["datetime_utc"] = (
        df["datetime_london"]
        .dt.tz_convert("UTC")
    )

    return df

labels = {"Nitric oxide":"NO", "Nitrogen dioxide":"NO2", "Nitrogen oxides as nitrogen dioxide":"NOx as NO2", "PM10 particulate matter (Hourly measured)":"PM10",
"PM2.5 particulate matter (Hourly measured)": "PM25", "Modelled Wind Direction": "WND", "Modelled Wind Speed":"WSP", "Modelled Temperature":"TMP",
"Ozone":"O3", "Sulphur dioxide":"SO2", "Carbon monoxide": "CO"}



In [ ]:
summary_file = Path("preprocessing_data_counts.csv")

def connect_db(dir_db):
    conn = sqlite3.connect(dir_db)
    query = "SELECT * FROM roadUK;"  
    traffic_df = pd.read_sql_query(query, conn)
    conn.close()
    traffic_df["geometry"] = traffic_df["geometry"].apply(lambda wkt: loads(wkt) if wkt else None)
    gdf = gpd.GeoDataFrame(traffic_df, geometry="geometry")
    gdf.set_crs("EPSG:4326", inplace=True) 
    return gdf

def aggregate_traffic_hourly_utc(df):
    """
    Aggregate traffic variables into hourly intervals defined in UTC.
    """
    data = df.dropna(subset=["datetime_utc"]).copy()

    # Assign each observation to the end of its UTC hourly interval.
    data["datetime"] = (
        data["datetime_utc"].dt.floor("h")
        + pd.Timedelta(hours=1)
    )

    required_columns = {
        "currentspeed",
        "freeflowspeed",
        "currenttraveltime",
        "freeflowtraveltime",
        "confidence",
    }

    for col in required_columns:
        data[col] = pd.to_numeric(
            data[col],
            errors="coerce"
        )

    data["traffic_level"] = (
        data["currentspeed"]
        / data["freeflowspeed"].replace(0, np.nan)
    )

    data["traveltime_level"] = (
        data["currenttraveltime"]
        / data["freeflowtraveltime"].replace(0, np.nan)
    )

    mean_traffic = (
        data
        .groupby("datetime")
        .agg({
            "currentspeed": "mean",
            "freeflowspeed": "mean",
            "traffic_level": "mean",
            "currenttraveltime": "mean",
            "freeflowtraveltime": "mean",
            "traveltime_level": "mean",
            "confidence": "mean",
            "roadclosure": (
                lambda s: (
                    s.mode().iloc[0]
                    if not s.mode().empty
                    else pd.NA
                )
            ),
        })
        .reset_index()
    )

    return mean_traffic


db_dir = r"road_traffic_TomTomUK.db"
stations_info = r"UK-air defra data"
stations = ["Camden - Euston Road", "Camden Kerbside", "London Marylebone Road", "Wandsworth - Putney High Street","Wandsworth - Putney",
            "Westminster - Oxford Street" ]
UK_air_id ={"Camden Kerbside":'UKA00259',
            "London Marylebone Road":'UKA00315', 
            "Wandsworth - Putney High Street":'WA8',
            "Wandsworth - Putney":'WA9',
            "Westminster - Oxford Street":"WM6",
            "Camden - Euston Road":'CD009'
            }


pollutant_stations = {
    "Camden - Euston Road": ["NO2"],
    "Camden Kerbside": ["NO2", "PM10", "PM25"],
    "London Marylebone Road": ["NO2", "PM10", "PM25", "O3"],
    "Wandsworth - Putney High Street": ["NO2", "PM10"],
    "Westminster - Oxford Street": ["NO2"]
}
preprocessing_counts = []
traffic_db = connect_db(db_dir)
for station in stations:
    
    traffic_id = UK_air_id[station]
    print(traffic_id, station)	
    
    filtered_traffic = traffic_db[ traffic_db["ukairid"] == traffic_id].copy()
    filtered_traffic.reset_index( drop=True, inplace=True )
    filtered_traffic = load_and_convert_traffic_to_utc(filtered_traffic)
    mean_traffic = aggregate_traffic_hourly_utc(filtered_traffic)
    mean_traffic.reset_index(drop=True, inplace=True)
    
    
    station_name = station + " 23jun2025-12dec2025.csv"
    df_pollutant = pd.read_csv(os.path.join(stations_info, station_name))
    df = df_pollutant.copy()
    df["datetime"] = parse_defra_datetime(df)
    keys = []
    for i in df.keys():
        if (not i.startswith("Status")) and i!="Station":
            keys.append(i)
    df = df[keys]
    df = df.replace(r'(?i)^no data$', np.nan, regex=True)
    cols = [df.columns[-1]] + list(df.columns[:-1])
    df = df[cols]
    df["datetime"] = pd.to_datetime(df["datetime"],errors="raise").dt.tz_localize("UTC")

    mean_traffic["datetime"] = pd.to_datetime( mean_traffic["datetime"],  errors="raise" )

    merged = pd.merge(df,mean_traffic, on="datetime", how="inner", validate="one_to_one")

    merged['year_month'] = merged['datetime'].dt.year.astype(str) +"_" + merged['datetime'].dt.month.astype(str)
    merged['hour'] = merged['datetime'].dt.hour
    merged['hour_sin'] = np.sin(2 * np.pi * merged['hour'] / 24)
    merged['hour_cos'] = np.cos(2 * np.pi * merged['hour'] / 24)
    merged.drop(columns=["Date", "Time"], inplace=True, errors='ignore')
    merged.rename(columns={"datetime":"date"}, inplace=True)
    
    
    merged = merged.rename(columns=labels)
    filename = station_name[:-4] +"_withtraffic.csv"
    merged.to_csv(os.path.join(stations_info,filename), index=False)
    
    print(len(df_pollutant))
    print(len(mean_traffic))
    if station != "Wandsworth - Putney":
        for pollutant in pollutant_stations[station]:
            preprocessing_counts.append({
                "station": station,
                "pollutant": pollutant,
                "n_pollution_raw": len(df),
                "n_traffic_hourly": len(mean_traffic),
                "n_after_traffic_pollution_merge": len(merged)
            })


df_preprocessing_counts = pd.DataFrame(preprocessing_counts)
df_preprocessing_counts.to_csv("preprocessing_data_counts.csv", index=False)


CD009 Camden - Euston Road
4152
4510
UKA00259 Camden Kerbside
4152
4706
UKA00315 London Marylebone Road
4152
4707
WA8 Wandsworth - Putney High Street
4152
4510
WA9 Wandsworth - Putney
4152
4510
WM6 Westminster - Oxford Street
4152
4510


In [3]:
df_preprocessing_counts

,station,pollutant,n_pollution_raw,n_traffic_hourly,n_after_traffic_pollution_merge
0,Camden - Euston Road,NO2,4152,4510,3747
1,Camden Kerbside,NO2,4152,4706,3943
2,Camden Kerbside,PM10,4152,4706,3943
3,Camden Kerbside,PM25,4152,4706,3943
4,London Marylebone Road,NO2,4152,4707,3944
5,London Marylebone Road,PM10,4152,4707,3944
6,London Marylebone Road,PM25,4152,4707,3944
7,London Marylebone Road,O3,4152,4707,3944
8,Wandsworth - Putney High Street,NO2,4152,4510,3747
9,Wandsworth - Putney High Street,PM10,4152,4510,3747


In [ ]:
stations_info = r"Bkg_or_with_traffic\raw_data_London"
stations = ["RBKC Knightsbridge (Kensington and Chelsea) 23jun2025-04mar2026.csv","Camden High Street 23jun2025-04mar2026.csv",
        "Background London Bloomsbury 23jun2025-03feb2026.csv", "Background London N. Kensington 23jun2025-03feb2026.csv", 
        "Background London Westminster 23jun2025-03feb2026.csv", "Wandsworth - Lavender Hill (Clapham Jct) 23jun2025-12mar2026.csv",
        "London Haringey Priory Park South 23jun2023-02mar2026.csv",
        "Brent - ARK Franklin Primary Academy.csv",
        'Richmond Upon Thames - Castelnau 23jun2025-12mar2026.csv']


for station_name in stations:
    df_pollutant = pd.read_csv(os.path.join(stations_info, station_name))
    df = df_pollutant.copy()
    df["datetime"] = parse_defra_datetime(df)
    keys = []
    for i in df.keys():
        if (not i.startswith("Status")) and i!="Station":
            keys.append(i)
    df = df[keys]
    df = df.replace(r'(?i)^no data$', np.nan, regex=True)
    cols = [df.columns[-1]] + list(df.columns[:-1])
    df = df[cols]
    
    df["datetime"] = pd.to_datetime(df["datetime"],errors="raise").dt.tz_localize("UTC")
    df.drop(columns=["Date", "Time"], inplace=True, errors='ignore')
    df.rename(columns={"datetime":"date"}, inplace=True)
        
    df = df.rename(columns=labels)

    output_filename = station_name.replace( ".csv", "_clean.csv" )

    df.to_csv(
        os.path.join(
            stations_info,
            output_filename
        ),
        index=False
    )